# Straight-through fake quantization with custom_vjp

Quantization-aware training runs the forward pass with weights rounded to a low-bit grid. Rounding has zero gradient almost everywhere, so training would stall. The **straight-through estimator** (STE) pretends rounding is the identity in the backward pass, and zeroes the gradient where the weight was clipped.

With `qmax` $= 2^{\text{bits}-1} - 1$, fake quantization is

$$\mathrm{fq}(w) = s \cdot \mathrm{clip}\bigl(\mathrm{round}(w / s),\, -q_{\max},\, q_{\max}\bigr)$$

Implement it with `jax.custom_vjp` so that its backward pass is

$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial\, \mathrm{fq}(w)} \cdot \mathbb{1}\bigl[\,|w / s| \le q_{\max}\bigr]$$

(the boundary $|w/s| = q_{\max}$ counts as inside). Rounding is half to even, as `jnp.round` does.

Write `qat_loss_and_grad(w, x, y, scale, bits)`:

- `w`: `(D, K)` float weights; `x`: `(B, D)`; `y`: `(B, K)`; `scale`: float $s > 0$; `bits`: int $\ge 2$.
- Loss: $\mathcal{L} = \mathrm{mean}\bigl((x\, \mathrm{fq}(w) - y)^2\bigr)$ over all $B \cdot K$ entries.

Return `(loss, grad_w, w_fq)`: float32 scalar, `(D, K)` STE gradient and `(D, K)` fake-quantized weights.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def qat_loss_and_grad(w, x, y, scale, bits):
    """Return (loss, STE gradient w.r.t. w (D, K), fake-quantized w (D, K))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.3], [-0.6]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, 0.5, 2], "e": {"__tup__": [{"__nd__": 0.0, "dtype": "float32"}, {"__nd__": [[0.0], [0.0]], "dtype": "float32"}, {"__nd__": [[0.5], [-0.5]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.5, 2.0], [-0.25, -3.0]], "dtype": "float64"}, {"__nd__": [[1.0, 2.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 0.5, 2], "e": {"__tup__": [{"__nd__": 0.6875, "dtype": "float32"}, {"__nd__": [[-0.25, 0.0], [-0.5, 0.0]], "dtype": "float32"}, {"__nd__": [[0.5, 0.5], [-0.0, -0.5]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.8267565599574231, -3.0783319101980338, 0.9580639753088469], [0.06963722766094482, 1.3182500241810684, 0.385629249998389], [1.8272586275861753, 0.0317437591517664, -0.5162294444924808], [0.5804849213397179, 0.43210686133773885, -0.35683935740335093]], "dtype": "float64"}, {"__nd__": [[-0.24730382198818454, 0.7194406781853278, 0.7043159938619936, -0.4939342302351804], [-0.3677137240199963, -1.8067903895865323, 1.6792074674884705, -0.2242908783928769], [1.337277430495411, 0.4174655938071864, 1.9439627746249157, 1.537109976128164], [0.318298439352904, 1.480763419858435, -0.9501235216099734, 1.2586181429890126], [-1.4804236275921896, 0.3432363675742628, 1.064876469210243, 0.22363214164877185]], "dtype": "float64"}, {"__nd__": [[-0.3671374972389881, -0.8055600446794365, -0.3428000151424468], [1.051125142503177, 0.8908394462458891, -0.2621314629648405], [-1.2460043473128628, 0.6739972418852872, -1.4498739364757562], [-0.530854904621622, -0.7348283771922636, 0.7432652102376149], [0.23594974138056524, 0.46185473227162593, 0.27240791046479607]], "dtype": "float64"}, 0.1, 4], "e": {"__tup__": [{"__nd__": 2.624098062515259, "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [0.5826685428619385, 0.0, 0.3629598915576935], [0.0, -0.6382341980934143, -0.3099145293235779], [1.0623762607574463, 0.20207636058330536, 0.17577847838401794]], "dtype": "float32"}, {"__nd__": [[0.699999988079071, -0.699999988079071, 0.699999988079071], [0.10000000149011612, 0.699999988079071, 0.4000000059604645], [0.699999988079071, 0.0, -0.5], [0.6000000238418579, 0.4000000059604645, -0.4000000059604645]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.6779301162200867, 0.535475983047878, 1.4124603684044394], [-0.03677124294459777, 0.6335944644542391, -0.125903192963739], [1.0285554273542712, 0.6666366521512817, 0.8758194707912133], [0.3484229955590589, 1.6400038821107275, -0.3611505786282483]], "dtype": "float64"}, {"__nd__": [[-0.33416607721805125, -0.5918599835407998, 0.610962701968474, -0.6224002707066544], [-0.6445326365001358, 0.7274748296203505, 1.1620957720506857, 0.5743514885621694], [-2.683103555466855, -0.9536324496718003, -1.07348948063148, -1.2075111797268347], [-0.4077676482371872, 1.8175432337026303, -0.3073985537659124, -0.7354457743627153], [0.6988227448389438, 0.06917172943950622, 0.12068112866070582, -0.6866741120790538]], "dtype": "float64"}, {"__nd__": [[-0.0015044670379158913, 0.43453717981416906, -1.9113065771604514], [-1.610224331010523, 1.0845605588518257, 0.8430045749240583], [-0.5066567175591229, -1.020203904417219, -0.18554662570969443], [-1.1936522904412012, -0.3400969906062227, 1.651100423975727], [0.1320718435065325, 1.1662884363547947, -0.22262052870934315]], "dtype": "float64"}, 0.02, 4], "e": {"__tup__": [{"__nd__": 1.1119928359985352, "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [0.3509407937526703, 0.0, -0.6985152363777161], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[-0.14000000059604645, 0.14000000059604645, 0.14000000059604645], [-0.03999999910593033, 0.14000000059604645, -0.11999999731779099], [0.14000000059604645, 0.14000000059604645, 0.14000000059604645], [0.14000000059604645, 0.14000000059604645, -0.14000000059604645]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.3694155619109898, 1.3858304929992982], [-1.3359375572222492, -0.34747799464982515], [-0.1568985005454348, 0.28198117936536454], [1.553666722804394, 0.6855270035798681], [-1.4976665985180717, -0.2709720411793551], [-0.8936434154830173, -0.919047029962896]], "dtype": "float64"}, {"__nd__": [[2.024827715003513, -1.0629706837475945, 0.23639517490996417, -0.7152122320787201, 0.8609809399519063, 0.35370323854134944], [0.1274277181041873, -0.07616714005920917, 1.1377169165755536, -0.9316998851391717, -0.7269383253215163, -1.1166060472205357], [1.334546412220603, -0.2935663794448887, -1.03740077070428, -0.4190734003279602, 2.499647141912049, -0.10747295455808797], [0.8500205977374702, 0.4651818264795043, -0.012383282080929063, -0.6927049513484409, -0.33429524042418257, -0.3544524150979252], [-2.209276174952708, 2.590132165299634, 0.9248362579408633, 0.11628650676709631, -0.20334913839455843, -0.7896443817288501], [0.949621715476415, -0.18685598546529084, 0.47927255908121963, -1.3874951680999685, 0.3784765367574817, 2.000301160167004], [-0.28962886468938337, 0.7999549486302511, 0.03533363711049965, 1.1051905403205458, 1.0674245093302333, 0.06164637112836233], [-0.559585430021553, 0.2867541614425335, 0.04176902945224784, -0.6179033056301174, 0.45647884748874545, 0.4323228618761901]], "dtype": "float64"}, {"__nd__": [[-0.8890266047821327, -0.1114878979154479], [2.921139507657893, 0.8168344872544508], [0.09777931084435634, 2.511082373702184], [-0.45540319859640854, -2.47685411921746], [-1.9069414995045355, -1.5195110137903052], [0.2600550542500467, 0.976356295342398], [-2.170190238668888, 1.1172437209364225], [-1.2277384405743208, 0.25643484822861606]], "dtype": "float64"}, 0.01, 8], "e": {"__tup__": [{"__nd__": 3.908764362335205, "dtype": "float32"}, {"__nd__": [[-0.38306647539138794, 0.0], [0.0, -0.510469377040863], [-0.3211464583873749, 0.04858297109603882], [0.0, -0.024049192667007446], [0.0, -0.9073397517204285], [-0.4992694854736328, -0.6991152763366699]], "dtype": "float32"}, {"__nd__": [[0.3700000047683716, 1.2699999809265137], [-1.2699999809265137, -0.3499999940395355], [-0.1599999964237213, 0.2800000011920929], [1.2699999809265137, 0.6899999976158142], [-1.2699999809265137, -0.26999998092651367], [-0.8899999856948853, -0.9199999570846558]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.3917588065221982, 1.2969895010717933, 0.022641240817181423], [0.5129865418276081, -3.419460449845638, 0.8346506711893347], [2.938957094277537, 2.778667892294288, 7.838558721216355]], "dtype": "float64"}, {"__nd__": [[1.0832325419401145, 0.7264273559501742, 1.6635005917178431], [0.24113763858859116, -1.2898772197929023, -2.23985985733975], [1.2765709163818362, 0.5680313489899385, 0.847296954372218], [-0.3328748962507801, 0.34482939616746583, 1.8823816030932006]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, 1.0, 2], "e": {"__tup__": [{"__nd__": 5.608579635620117, "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.46717995405197144], [1.4911991357803345, 0.0, 1.3101658821105957], [0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 0.0], [1.0, -1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.016741710092576738, -0.05577593381339044, -0.02157248578173418, 0.0692555307793498], [-0.0544980532143469, -0.1290714730009565, -0.11469775939727436, -0.06694694793093219], [0.039969676424662634, 0.004423160046367068, -0.037863856357963854, -0.06425682647092174], [-0.005699465406653111, -0.007979781652170106, -0.02819634720256439, -0.012262707302620239], [-0.03815908886074152, 0.0901610970292781, -0.11834706827749944, 0.0061094427982471625], [-0.050720278725867755, 0.03532582192512178, -0.01175530986868252, 0.01830107236240623], [0.026579812191453202, 0.010266311919835746, 0.049552023087200796, -0.0695738877231753], [0.04758494560820884, -0.0109471224812893, 0.01903221886619394, 0.07782666870489997]], "dtype": "float64"}, {"__nd__": [[1.124406837228961, 1.983906260402673, -2.250591833413329, -0.9558816529403455, 0.9641676674228251, -1.130678125936227, -0.18259281381333614, 1.7078758612485556], [0.8746999899786561, -0.9622686699630217, 0.4538064407814815, 0.6453005002956944, 0.06737638601500909, -0.10050634112829009, 0.1415574756124382, -1.5005136959043504], [0.12651241840846753, -0.9669110793525206, 0.39357107135464237, 0.42921695904368384, -0.892793268783839, 0.612671144108547, 0.3558124292565846, -0.04209628672662296], [0.7258067220827112, -0.31279669658828074, -0.28415798222731575, 0.07778617965586546, -0.059222803868094115, 1.142166871287435, -2.373945235389359, -0.3104735529634693], [-2.0149561047575175, -0.5262386979768381, 0.2322282371772858, -0.022146193899912747, 0.9393318676256391, 0.2866276607088688, -2.7768745606713137, -1.1583429548412214], [1.5136624934334706, 1.5159204065480982, 0.05234707902313534, -1.601787701176551, 2.118804070102688, -2.1530353066668972, 0.25024316589593604, -0.11187308559505371], [0.17806975537603983, -1.240717357266528, -0.4722427659931478, 1.0021493076522272, -0.566389865708578, -1.4614338387338917, -0.09685989180754231, 0.006268835734178283], [-0.22062527245614327, -0.8773841377983873, 0.4550834628751101, -0.21320793667014518, -1.2648979159843472, 0.13264485194458006, -0.11691341642839587, 0.05893535902297446], [-1.7931824180543972, -0.8577885375379697, 0.37299983596970493, 0.8514968812412, 0.26848824196572596, 0.08492853776235817, 0.6574750997519162, 0.9197622559268187], [1.7089095991597132, 0.416161651662968, -0.617898809061064, 0.43328855938185223, -1.994313337409666, 0.02022833988314914, -0.4773692452515349, -0.05159262496450286], [0.46636249718118594, 0.028592398183438686, -1.472240175171709, -0.3766682516719827, -0.8190875247325766, -0.035214233348543234, 0.34775832505667026, -0.9762750527390194], [0.7236144996200846, 1.2932844357111286, -0.2788844254279498, 0.7214635410570496, -1.4895076996078205, -0.5475362446940503, 0.9577655289636221, -1.771726555102153], [-0.5123825891381801, -0.9021143736434656, 1.1438186127113699, 1.1407461185846095, -0.6291699856677592, 0.11823820239791931, -2.6185953294711948, 0.8450719508280701], [0.34489307430605387, 1.321676514269612, -2.3936133348335895, 0.34245155509221265, -0.3584923321792362, -2.617582772625446, -0.6984831500347612, -0.08622462961584804], [0.7974617928943865, 0.19020103547696696, -0.5223810280628554, 0.43909767776573405, -0.11440622715817586, -0.6565077024775354, -1.0265801957742764, 0.1428425333688465], [0.15089504441394663, 1.835772183308189, -0.45719439925974104, 1.599266519377065, 0.3460621048198438, 1.173685962859027, -0.9101956095226067, -0.8784043385444784]], "dtype": "float64"}, {"__nd__": [[-0.3121608183309457, 0.8927077912890001, 0.17324927936697432, 0.750797171358702], [0.3159587283344176, 1.0272805308398631, 0.19700369537289608, -0.526489476375156], [0.5140479005699569, 0.09003370802947244, 1.7483366472512227, -0.15972213232813467], [0.10902445224740528, -0.3064395127632179, 1.1561104824845763, 0.8746158488269457], [0.6510992265369949, -1.2466926303677017, -1.5165450265163778, 0.2648726884435707], [0.7220378712890259, 1.4791980565820146, -1.7956932334440707, -0.17030522858961525], [-1.6904635961913856, -0.9607281602137127, -0.2211023076367421, 0.9630199134878099], [-0.5266167805021937, -0.718177505157109, -0.3198303871933791, 1.2576661536807427], [2.415912944780984, -1.5420102096218067, -0.27349778189704577, -0.43029893858656654], [1.1688016828523946, 0.4356662067659889, 1.5407292766608034, -1.0121275213254168], [0.013772513416587234, -0.45555510947930655, -1.0964166634549601, -0.15803399702278426], [-1.1844993826725632, 2.1726217855985626, 0.417291116991025, -1.0164598086367858], [-0.6060484283385521, 1.7326364891299604, -0.4781183586479403, 0.6941054669674107], [0.5836544246629394, 1.0927316672741476, 0.6347154356607344, -0.21011425768109862], [-0.5914984158560455, -1.7757763891268423, -0.047652759873061314, 0.13870155113204108], [-0.36019461661277574, -0.1436373634512495, -1.2139527451556316, 2.171170181466329]], "dtype": "float64"}, 0.0071428571428571435, 4], "e": {"__tup__": [{"__nd__": 1.0175307989120483, "dtype": "float32"}, {"__nd__": [[0.10853958129882812, 0.0, -0.15805931389331818, 0.0], [0.0, 0.0, 0.0, 0.0], [-0.0012705485569313169, 0.10427476465702057, 0.04286525771021843, 0.0], [0.04870406910777092, 0.04511196166276932, -0.07984469085931778, -0.08709946274757385], [-0.11654561758041382, 0.0, 0.0, -0.07817995548248291], [0.0, 0.24480779469013214, -0.08900744467973709, -0.09289869666099548], [0.007720041088759899, -0.07266032695770264, -0.04603175073862076, 0.0], [-0.04217154160141945, 0.07190082967281342, -0.055366091430187225, 0.0]], "dtype": "float32"}, {"__nd__": [[0.014285714365541935, -0.05000000074505806, -0.02142857201397419, 0.05000000074505806], [-0.05000000074505806, -0.05000000074505806, -0.05000000074505806, -0.05000000074505806], [0.04285714402794838, 0.0071428571827709675, -0.0357142873108387, -0.05000000074505806], [-0.0071428571827709675, -0.0071428571827709675, -0.02857142873108387, -0.014285714365541935], [-0.0357142873108387, 0.05000000074505806, -0.05000000074505806, 0.0071428571827709675], [-0.05000000074505806, 0.0357142873108387, -0.014285714365541935, 0.02142857201397419], [0.02857142873108387, 0.0071428571827709675, 0.05000000074505806, -0.05000000074505806], [0.05000000074505806, -0.014285714365541935, 0.02142857201397419, 0.05000000074505806]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(qat_loss_and_grad, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")